In [9]:
from google.colab import drive
from datasets import load_from_disk
import pandas as pd

drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_Justin.csv"

combined_path = "/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/combined_datasets_lite_hf"

combined_dataset = load_from_disk(combined_path)

df_combined = combined_dataset.to_pandas()
print(df_combined.head())


existing_path = "/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/MICE_Output/eval-1k/summary_full.csv"
existing_df = pd.read_csv(existing_path)
print(existing_df.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
                                            question  \
0  The actor who played Jack Geller in Friends wa...   
1  When will dragon ball super english dub be rel...   
2  In interior design what kind of complementary ...   
3            Golf star Vijay Singh comes form where?   
4       Where does the grand canal start and finish?   

                                   answer  \
0  Elliott Gould married Barbra Streisand   
1                                    2017   
2                                     N/A   
3                                    Fiji   
4                    Dublin , in the east   

                            question_id     type  
0  88e595ba-722f-4666-bfe3-2bf33201c142   answer  
1  6fde1caf-67b9-479b-b017-9c15f6117a0d  clarify  
2  dceab11a-be41-48de-abd5-59266b432b6e  abstain  
3  ac3b0a3b-3194-42c3-9e2b-c3a6a19beee0   answer  
4  f969c

In [10]:
# Duplicate check on 'question'
dup_question_mask = existing_df.duplicated(subset='question', keep=False)
n_dup_questions = dup_question_mask.sum()
print(f"Duplicate 'question' rows in existing_df: {n_dup_questions}")
if n_dup_questions > 0:
    print(existing_df[dup_question_mask][['question', 'question_id', 'type']].sort_values('question'))

# Duplicate check on 'question_id'
dup_qid_mask = existing_df.duplicated(subset='question_id', keep=False)
n_dup_qids = dup_qid_mask.sum()
print(f"\nDuplicate 'question_id' rows in existing_df: {n_dup_qids}")
if n_dup_qids > 0:
    print(existing_df[dup_qid_mask][['question_id', 'question', 'type']].sort_values('question_id'))

# Type breakdown
print("\n── existing_df 'type' breakdown ──")
print(existing_df['type'].value_counts())

Duplicate 'question' rows in existing_df: 0

Duplicate 'question_id' rows in existing_df: 0

── existing_df 'type' breakdown ──
type
abstain    334
answer     333
clarify    333
Name: count, dtype: int64


In [11]:
SAMPLE_PER_TYPE = 500
TARGET_TYPES    = ['abstain', 'clarify', 'answer']

# Build exclusion sets for both question text and question_id
existing_questions = set(existing_df['question'].dropna().str.strip())
existing_qids      = set(existing_df['question_id'].dropna())

# Exclude rows where either 'question' or 'question_id' already exists in existing_df
df_pool = df_combined[
    ~df_combined['question'].str.strip().isin(existing_questions) &
    ~df_combined['question_id'].isin(existing_qids)
].copy()

print(f"Pool size after removing existing questions & IDs: {len(df_pool):,}")
print("Pool type breakdown:\n", df_pool['type'].value_counts())

# Sample 500 per type (raises an error early if a type is under-represented)
sampled_parts = []
for t in TARGET_TYPES:
    pool_t = df_pool[df_pool['type'] == t]
    if len(pool_t) < SAMPLE_PER_TYPE:
        raise ValueError(
            f"Not enough '{t}' rows in pool: need {SAMPLE_PER_TYPE}, have {len(pool_t)}"
        )
    sampled_parts.append(pool_t.sample(n=SAMPLE_PER_TYPE, random_state=42))

df_new = pd.concat(sampled_parts, ignore_index=True)
print(f"\nSampled df_new shape : {df_new.shape}")
print("df_new 'type' breakdown:\n", df_new['type'].value_counts())

Pool size after removing existing questions & IDs: 2,156
Pool type breakdown:
 type
abstain    820
answer     669
clarify    667
Name: count, dtype: int64

Sampled df_new shape : (1500, 4)
df_new 'type' breakdown:
 type
abstain    500
clarify    500
answer     500
Name: count, dtype: int64


In [12]:
KEEP_COLS = ['question', 'answer', 'type', 'question_id']
df_new = df_new[KEEP_COLS].reset_index(drop=True)

df_new.to_csv(save_path, index=False)
print(f"Saved {len(df_new):,} rows → {save_path}")
print(df_new.head())

Saved 1,500 rows → /content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_Justin.csv
                                            question answer     type  \
0  How much longer are we going to be on this pla...    N/A  abstain   
1  Frank was reading through his favorite book. H...    N/A  abstain   
2  Would motion graphics in film be better if the...    N/A  abstain   
3  How many total action figures will he have by ...    N/A  abstain   
4       How many outfits does she have for her baby?    N/A  abstain   

                            question_id  
0  18c556af-e56a-4beb-8df5-fc7224e806c4  
1  5235d7c0-c3b9-4bf0-9741-97484ae871bc  
2  405b9e06-d67f-4cb7-8ff9-3f0684056e7b  
3  0a91c800-6a31-4696-941b-ba548dd00069  
4  d993b6a7-e59c-43f7-a09f-68eabd6e01f2  


In [13]:
overlap_questions = set(existing_df['question'].dropna().str.strip()) & \
                    set(df_new['question'].dropna().str.strip())
print(f"Shared 'question' values between existing_df & df_new: {len(overlap_questions)}")
if overlap_questions:
    print("  Examples:", list(overlap_questions)[:5])

if 'question_id' in existing_df.columns:
    overlap_ids = set(existing_df['question_id'].dropna()) & \
                  set(df_new['question_id'].dropna())
    print(f"Shared 'question_id' values between existing_df & df_new: {len(overlap_ids)}")
    if overlap_ids:
        print("  Examples:", list(overlap_ids)[:5])
else:
    print("'question_id' column not present in existing_df — skipping id overlap check.")


Shared 'question' values between existing_df & df_new: 0
Shared 'question_id' values between existing_df & df_new: 0


In [14]:
df_combined_audit = pd.concat(
    [existing_df[['question', 'type']].assign(source='existing'),
     df_new[['question', 'type']].assign(source='new')],
    ignore_index=True
)

print("\n── Combined 'type' breakdown (existing + new) ──")
print(df_combined_audit['type'].value_counts())

print("\n── Per-source 'type' breakdown ──")
print(df_combined_audit.groupby(['source', 'type']).size().unstack(fill_value=0))


── Combined 'type' breakdown (existing + new) ──
type
abstain    834
answer     833
clarify    833
Name: count, dtype: int64

── Per-source 'type' breakdown ──
type      abstain  answer  clarify
source                            
existing      334     333      333
new           500     500      500
